# 05 — Hybrid recommender (Improved system)

**Phase 2.4** — Combines ingredient match, SVD prediction, expiry priority, nutrition, and CARS context boosts.

**Warm user formula:** `0.35×match + 0.30×CF + 0.20×expiry + 0.15×nutrition`

**Cold-start formula:** `0.50×match + 0.30×expiry + 0.20×nutrition`

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data_loader import load_fridgewise_data
from src.models.collaborative import CollaborativeRecommender
from src.models.content_based import ContentBasedRecommender
from src.recommender import HybridRecommender

In [ ]:
data = load_fridgewise_data(ROOT)

cf = CollaborativeRecommender(n_factors=50, n_epochs=20).fit(data)
content = ContentBasedRecommender().fit(data)
hybrid = HybridRecommender().fit(data, cf, content)
hybrid.save(ROOT / "src" / "models" / "artifacts" / "hybrid.pkl")

In [ ]:
import pandas as pd

demo_user = int(data.profiles.iloc[0]["user_id"])
profile = data.profiles[data.profiles["user_id"] == demo_user].iloc[0]
print(f"User {demo_user} | diet: {profile['dietary_type']} | allergies: {profile['allergies']}")

recs = hybrid.recommend(demo_user, k=10)
pd.DataFrame([{"rank": i+1, "recipe_id": r.recipe_id, "recipe_name": r.recipe_name, "hybrid_score": round(r.score, 3)}
              for i, r in enumerate(recs)])

## Compare all four models

Or run `python scripts/train_models.py` to train and save all models at once.

In [ ]:
from src.models.popularity import PopularityRecommender

pop = PopularityRecommender().fit(data)
warm_user = int(data.interactions["user_id"].value_counts().index[0])

for name, model, uid in [
    ("Popularity", pop, warm_user),
    ("Content", content, demo_user),
    ("SVD", cf, warm_user),
    ("Hybrid", hybrid, demo_user),
]:
    top = model.recommend(uid, k=3)
    print(f"\n{name} (user {uid}):")
    for r in top:
        print(f"  {r.score:.3f}  {r.recipe_name[:55]}")